# 🍛 RecipeGPT v2 — High Quality Indian Recipe Generator

**Upgrades over v1:**
- 5× larger corpus (~250K chars) with 50+ detailed recipes
- Bigger model: 8 layers, 8 heads, 512 embedding dim (~30M params)
- 10,000 training steps with cosine warmup
- Structured recipe format enforced in training data
- Beam-search style top-p (nucleus) sampling for coherent output
- Gradient accumulation for effective larger batch size


## 1. Install & Imports

In [ ]:
# Run this cell first
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import matplotlib.pyplot as plt
import time, math, random

print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 2. Large Recipe Corpus (~250K chars)

In [ ]:
# ============================================================
# CELL 2: Expanded Recipe Corpus
# Each recipe has strict structure: NAME / CATEGORY / INGREDIENTS / METHOD / TIPS
# This structure helps the model learn recipe format deeply
# ============================================================

BASE_CORPUS = '''
[RECIPE]
NAME: Butter Chicken (Murgh Makhani)
CATEGORY: Main Course, North Indian
SERVES: 4
TIME: 45 minutes
INGREDIENTS:
- 600g chicken thighs, boneless, cut into cubes
- 3 tbsp butter
- 2 tbsp oil
- 1.5 cups tomato puree
- 3/4 cup heavy cream
- 2 tsp kashmiri red chili powder
- 1.5 tsp garam masala
- 1 tsp cumin powder
- 1 tsp coriander powder
- 1/2 tsp turmeric powder
- 1 tbsp ginger garlic paste
- 1 tbsp kasuri methi (dried fenugreek leaves)
- 1 tsp sugar
- Salt to taste
MARINADE:
- 1/2 cup yogurt
- 1 tsp red chili powder
- 1 tsp garam masala
- 1 tbsp lemon juice
- 1 tbsp oil
- Salt to taste
METHOD:
1. Mix all marinade ingredients, coat chicken well and refrigerate for 4 hours or overnight.
2. Grill or pan-fry chicken pieces on high heat until slightly charred on edges. Set aside.
3. In a heavy pan, heat butter and oil together over medium heat.
4. Add ginger garlic paste and saute for 2 minutes until raw smell disappears.
5. Add tomato puree and cook on medium heat for 12 to 15 minutes until oil separates.
6. Add kashmiri chili, garam masala, cumin, coriander and turmeric. Cook 3 minutes.
7. Reduce heat, add cream and sugar. Stir well and simmer for 5 minutes.
8. Add grilled chicken pieces, kasuri methi and salt. Simmer 10 minutes.
9. Adjust consistency with water if needed. Sauce should coat the back of a spoon.
TIPS: The key to restaurant-style butter chicken is the kasuri methi added at the end.
Resting the finished curry for 10 minutes before serving deepens the flavor significantly.
Serve with butter naan, garlic naan, or steamed basmati rice.
[/RECIPE]

[RECIPE]
NAME: Dal Makhani
CATEGORY: Main Course, North Indian, Vegetarian
SERVES: 4
TIME: 8 hours (including soaking)
INGREDIENTS:
- 1 cup whole black lentils (sabut urad dal)
- 1/4 cup red kidney beans (rajma)
- 4 tbsp butter
- 2 tbsp oil
- 1.5 cups tomato puree
- 3/4 cup fresh cream
- 1 large onion, finely chopped
- 1 tbsp ginger garlic paste
- 1 tsp cumin seeds
- 1.5 tsp coriander powder
- 1 tsp red chili powder
- 1 tsp garam masala
- 1/2 tsp turmeric
- Salt to taste
METHOD:
1. Soak urad dal and rajma together in water overnight, minimum 8 hours.
2. Pressure cook with 3 cups water, salt and turmeric for 10 whistles.
3. Dal should be completely soft. Mash about 1/4 of the dal with the back of a spoon.
4. In a separate pan, heat butter and oil. Add cumin seeds and let them splutter.
5. Add onions and fry on medium heat until deep golden brown, about 15 minutes.
6. Add ginger garlic paste, cook 3 minutes. Add tomato puree, cook until oil separates.
7. Add all dry spices except garam masala. Cook masala for 5 minutes.
8. Add cooked dal to masala. Mix well. Add 1/2 cup water.
9. Simmer on very low heat for 45 minutes to 1 hour, stirring every 10 minutes.
10. Add cream and garam masala in last 10 minutes. Adjust salt.
TIPS: The secret to great dal makhani is long slow cooking. The longer it simmers,
the more the flavors develop and the creamier it becomes. Overnight slow cooking
on a wood fire is the traditional method. Finish with a generous dollop of butter.
Serve with naan, paratha, or steamed rice.
[/RECIPE]

[RECIPE]
NAME: Hyderabadi Chicken Biryani
CATEGORY: Main Course, Hyderabadi
SERVES: 6
TIME: 2 hours
INGREDIENTS:
- 1 kg chicken, cut into large pieces with bone
- 3 cups basmati rice, aged
- 3 large onions, thinly sliced
- 1.5 cups yogurt
- 1/2 cup oil for frying onions
- 4 tbsp ghee
- 2 tbsp biryani masala
- 2 tsp red chili powder
- 1 tsp turmeric
- 2 tbsp ginger garlic paste
- Large handful fresh mint leaves
- Large handful fresh coriander leaves
- Generous pinch saffron soaked in 4 tbsp warm milk
- 2 tbsp rose water
- Whole spices: 4 bay leaves, 6 cardamom, 6 cloves, 2 inch cinnamon, 1 tsp black pepper
- Salt to taste
METHOD:
1. Fry sliced onions in oil on medium heat until deep golden brown and crispy. Drain and set aside.
2. Marinate chicken with yogurt, biryani masala, chili, turmeric, ginger garlic paste,
   half the fried onions, half the mint, half the coriander, salt. Rest 2 hours minimum.
3. Wash basmati rice 3 times until water runs clear. Soak 30 minutes.
4. Boil water with whole spices, salt, 1 tbsp oil. Add soaked rice.
5. Cook rice until 70 percent done. Grains should have a bite. Drain immediately.
6. In a heavy bottomed pot, spread marinated chicken as first layer.
7. Layer partially cooked rice evenly on top of chicken.
8. Sprinkle remaining fried onions, mint, coriander over rice.
9. Drizzle saffron milk, rose water and ghee over entire surface.
10. Seal pot tightly with foil then lid. Cook on high heat 5 minutes.
11. Reduce to lowest heat possible and cook on dum for 25 minutes.
12. Rest sealed pot 10 minutes before opening. Mix gently from bottom.
TIPS: The dum cooking is non-negotiable for authentic biryani. Use a tawa (griddle)
under the pot to prevent burning. Aged basmati rice gives longer, non-sticky grains.
Serve with mirchi ka salan, raita and a squeeze of lime.
[/RECIPE]

[RECIPE]
NAME: Palak Paneer
CATEGORY: Main Course, North Indian, Vegetarian
SERVES: 4
TIME: 30 minutes
INGREDIENTS:
- 300g paneer, cut into 1 inch cubes
- 600g fresh spinach leaves
- 2 medium tomatoes, roughly chopped
- 1 large onion, chopped
- 4 cloves garlic
- 1 inch fresh ginger
- 3 green chilies
- 3 tbsp oil or ghee
- 1/2 cup cream
- 1 tsp cumin seeds
- 1 tsp coriander powder
- 1/2 tsp turmeric
- 1 tsp garam masala
- 1 tbsp kasuri methi
- Salt to taste
METHOD:
1. Blanch spinach in boiling salted water for exactly 2 minutes. Transfer to ice cold water.
2. Blend spinach with green chilies until very smooth. Set aside.
3. Fry paneer cubes in oil until golden on all sides. Remove and soak in warm salted water.
4. In same pan add cumin seeds. Add onion and cook until golden.
5. Add garlic and ginger paste, cook 2 minutes. Add tomatoes, cook until mushy.
6. Add coriander powder and turmeric. Cook masala 3 minutes.
7. Add spinach puree, mix well. Cook on medium heat 5 minutes.
8. Add cream and kasuri methi. Stir gently.
9. Add soaked paneer cubes. Season with garam masala and salt.
10. Simmer 5 minutes. Do not overcook or spinach loses its green color.
TIPS: The ice bath after blanching is what keeps spinach vibrantly green.
Soaking fried paneer in warm water keeps it soft and prevents rubbery texture.
For a richer version, add 2 tbsp fresh cream swirled on top before serving.
[/RECIPE]

[RECIPE]
NAME: Chole Masala (Punjabi Chickpea Curry)
CATEGORY: Main Course, Punjabi, Vegetarian
SERVES: 4
TIME: 1 hour
INGREDIENTS:
- 2 cups dried chickpeas
- 3 medium onions, finely chopped
- 3 large tomatoes, pureed
- 2 tbsp ginger garlic paste
- 3 tbsp oil
- 2 tsp chole masala powder
- 1.5 tsp coriander powder
- 1 tsp cumin powder
- 1 tsp red chili powder
- 1/2 tsp turmeric
- 1 tsp amchur (dry mango powder)
- 1 tsp garam masala
- 2 tea bags (for dark color)
- 1 tsp cumin seeds
- Fresh coriander for garnish
- Salt to taste
METHOD:
1. Soak chickpeas overnight in plenty of water.
2. Pressure cook chickpeas with tea bags and salt for 6 to 8 whistles until very soft.
3. Discard tea bags. Reserve the cooking liquid.
4. Heat oil in a heavy pan. Add cumin seeds.
5. Add onions and cook on medium heat until deep golden brown, about 20 minutes.
6. Add ginger garlic paste, cook until raw smell gone, about 3 minutes.
7. Add tomato puree, cook on high heat for 10 minutes until oil separates.
8. Add all dry spices except amchur and garam masala. Cook 3 minutes.
9. Add cooked chickpeas with about 1 cup reserved liquid.
10. Mash about 1/4 of the chickpeas with back of spoon to thicken gravy.
11. Simmer 15 minutes. Add amchur, garam masala and fresh coriander.
TIPS: Tea bags give chole its characteristic dark brown color without affecting flavor.
Amchur added at the end gives the authentic tangy punch. Never skip it.
Serve with bhature, puri, or jeera rice.
[/RECIPE]

[RECIPE]
NAME: Paneer Butter Masala
CATEGORY: Main Course, North Indian, Vegetarian
SERVES: 4
TIME: 35 minutes
INGREDIENTS:
- 400g fresh paneer, cubed
- 4 large tomatoes
- 2 medium onions, roughly chopped
- 8 cashews
- 4 cloves garlic
- 1 inch ginger
- 3 tbsp butter
- 1 tbsp oil
- 1/2 cup cream
- 2 tsp kashmiri chili powder
- 1.5 tsp coriander powder
- 1 tsp garam masala
- 1 tsp sugar
- 1 tbsp kasuri methi
- Salt to taste
METHOD:
1. Saute onions, tomatoes, garlic, ginger and cashews together until soft.
2. Cool completely and blend to a very smooth paste. Pass through a fine strainer.
3. Heat butter and oil in pan. Pour strained tomato puree carefully.
4. Cook on medium heat for 15 minutes, stirring regularly, until oil appears on surface.
5. Add kashmiri chili and coriander powder. Cook 3 minutes.
6. Add cream and sugar. Stir and simmer 5 minutes on low heat.
7. Add paneer cubes gently. Fold in without breaking pieces.
8. Add kasuri methi crushed between palms, garam masala and salt.
9. Simmer 5 minutes. Finish with 1 tsp butter on top.
TIPS: Straining the blended gravy is what gives restaurant-quality smooth texture.
Cashews add body and natural creaminess to the sauce.
Kashmiri chili gives vibrant color without excessive heat.
[/RECIPE]

[RECIPE]
NAME: Lamb Rogan Josh
CATEGORY: Main Course, Kashmiri
SERVES: 4
TIME: 1.5 hours
INGREDIENTS:
- 800g lamb shoulder, bone-in, cut into pieces
- 4 tbsp mustard oil
- 3 large onions, thinly sliced
- 1 cup yogurt, whisked
- 2 tbsp Kashmiri red chili powder
- 2 tsp fennel powder (saunf)
- 1 tsp dry ginger powder (sonth)
- 1/2 tsp cardamom powder
- 1 tsp garam masala
- Whole spices: 4 cloves, 4 cardamom, 2 bay leaves, 1 stick cinnamon
- 2 tsp ratan jot (optional, for authentic red color)
- Salt to taste
METHOD:
1. Heat mustard oil until it starts smoking. Remove from heat for 30 seconds.
2. Return to medium heat. Add whole spices and let them crackle.
3. Add onions and cook until dark golden, about 25 minutes.
4. Increase heat. Add lamb pieces and brown on all sides for 10 minutes.
5. Reduce heat. Add Kashmiri chili powder and cook 2 minutes.
6. Add whisked yogurt one tablespoon at a time, stirring constantly to prevent curdling.
7. Once all yogurt incorporated, add fennel powder, ginger powder and cardamom.
8. Add 1 cup hot water. Cover and simmer for 45 minutes until lamb is very tender.
9. Add garam masala and salt. Cook uncovered 10 minutes to reduce sauce.
TIPS: Smoking the mustard oil first removes its pungency and is essential for authentic flavor.
Adding yogurt slowly prevents it from splitting in the hot oil.
Rogan Josh should have a thick, coating gravy, not a thin sauce.
[/RECIPE]

[RECIPE]
NAME: Masala Dosa with Sambar
CATEGORY: South Indian, Breakfast
SERVES: 4
TIME: 12 hours (including fermentation)
INGREDIENTS FOR DOSA BATTER:
- 3 cups raw rice
- 1 cup urad dal
- 1/4 cup chana dal
- 1 tsp fenugreek seeds
- Salt to taste
INGREDIENTS FOR POTATO MASALA FILLING:
- 4 large potatoes, boiled and roughly mashed
- 2 onions, sliced
- 3 green chilies, slit
- 1 inch ginger, grated
- 2 tbsp oil
- 1 tsp mustard seeds
- 1 tsp turmeric
- 10 curry leaves
- 2 tbsp fresh coriander
- Salt to taste
INGREDIENTS FOR SAMBAR:
- 1 cup toor dal
- 2 cups mixed vegetables (drumstick, eggplant, shallots, tomato)
- 2 tbsp sambar powder
- Lemon sized tamarind ball, soaked in warm water
- 1 tsp mustard seeds
- 10 curry leaves
- 2 dried red chilies
- 1/4 tsp asafoetida (hing)
- 2 tbsp oil
- Salt to taste
METHOD FOR BATTER:
1. Soak rice, urad dal, chana dal and fenugreek seeds separately for 6 hours.
2. Grind urad dal with minimal water to very smooth fluffy batter.
3. Grind rice to slightly coarse batter separately.
4. Mix both batters together. Add salt. Batter should be pourable consistency.
5. Ferment in warm place for 8 to 12 hours until batter doubles and smells sour.
METHOD FOR SAMBAR:
1. Cook toor dal with turmeric until completely mushy.
2. Add tamarind water, sambar powder, vegetables and salt. Simmer 15 minutes.
3. Prepare tempering: heat oil, add mustard seeds, curry leaves, dried chilies, hing.
4. Pour tempering into sambar. Simmer 5 more minutes.
METHOD FOR MASALA FILLING:
1. Heat oil, add mustard seeds and curry leaves.
2. Add onions, green chilies and ginger. Cook until soft.
3. Add turmeric and mashed potatoes. Mix well. Season with salt and coriander.
METHOD FOR DOSA:
1. Heat cast iron griddle until very hot. Sprinkle few drops water, it should evaporate instantly.
2. Pour a ladleful of batter in center. Spread in concentric circles to thin crepe.
3. Drizzle 1 tsp oil around edges and center. Cook until edges turn crispy and brown.
4. Place 2 tbsp potato filling on one side. Fold dosa over filling.
5. Serve immediately with sambar and coconut chutney.
TIPS: Well fermented batter is the foundation of good dosa. The sour smell indicates
proper fermentation. Spreading batter quickly in one motion prevents tearing.
[/RECIPE]

[RECIPE]
NAME: Rajma (Red Kidney Bean Curry)
CATEGORY: Main Course, Punjabi, Vegetarian
SERVES: 4
TIME: 1 hour
INGREDIENTS:
- 2 cups dried red kidney beans (rajma)
- 3 medium onions, finely chopped
- 3 large tomatoes, pureed
- 2 tbsp ginger garlic paste
- 3 tbsp oil or ghee
- 1 tsp cumin seeds
- 2 tsp coriander powder
- 1.5 tsp red chili powder
- 1/2 tsp turmeric
- 1 tsp garam masala
- 1/2 tsp amchur
- Fresh coriander leaves
- Salt to taste
METHOD:
1. Soak rajma overnight. Pressure cook with salt for 8 whistles until very tender.
2. Beans should be soft enough to mash easily between fingers.
3. Heat ghee, add cumin seeds. Add onions and cook until deep golden, 20 minutes.
4. Add ginger garlic paste, cook 3 minutes.
5. Add tomato puree and cook on high heat until oil separates, about 12 minutes.
6. Add coriander powder, chili powder, turmeric. Cook 3 minutes.
7. Add cooked rajma with all its cooking liquid.
8. Mash about 1/3 of the beans to naturally thicken the gravy.
9. Simmer 20 minutes on low heat. Add amchur and garam masala.
10. Finish with fresh coriander and a drizzle of ghee.
TIPS: The mashing of some beans is essential for the thick, gravy-coating texture.
Rajma chawal (with steamed rice) is a complete protein and the ultimate Punjabi comfort food.
[/RECIPE]

[RECIPE]
NAME: Gulab Jamun
CATEGORY: Dessert, North Indian
SERVES: 20 pieces
TIME: 45 minutes
INGREDIENTS FOR JAMUN:
- 1 cup full fat milk powder
- 3 tbsp all purpose flour (maida)
- 1/4 tsp baking soda
- 2 tbsp ghee, softened
- 3 to 4 tbsp warm milk for binding
- Oil for deep frying
INGREDIENTS FOR SUGAR SYRUP:
- 2 cups sugar
- 1.5 cups water
- 4 cardamom pods, crushed
- Generous pinch saffron
- 1 tsp rose water
- 1/4 tsp lemon juice
METHOD:
1. Make sugar syrup first: combine sugar, water, cardamom and saffron.
2. Heat until sugar dissolves. Add lemon juice to prevent crystallization.
3. Simmer until slightly sticky, about 10 minutes. Add rose water. Keep warm.
4. For jamuns: mix milk powder, flour, baking soda together.
5. Rub in softened ghee until mixture resembles breadcrumbs.
6. Add warm milk gradually, just enough to form a soft, non-sticky dough.
7. Do not knead aggressively. Rest dough 5 minutes.
8. Divide into 20 equal portions. Roll into smooth crack-free balls.
9. Heat oil to medium-low (160 degrees). Fry balls in small batches.
10. Fry very slowly for 8 to 10 minutes, turning constantly until deep brown.
11. Remove and immediately add to warm sugar syrup.
12. Soak for minimum 2 hours. Jamuns will double in size.
TIPS: Frying on low heat is the most critical step. High heat burns outside
while inside stays raw. Crack-free balls are essential, any crack will cause
the jamun to break open during frying. Syrup must be warm when jamuns are added.
[/RECIPE]

[RECIPE]
NAME: Chicken Tikka Masala
CATEGORY: Main Course, North Indian
SERVES: 4
TIME: 1 hour
INGREDIENTS:
- 700g boneless chicken breast or thigh
- 3 tbsp butter
- 1 tbsp oil
- 2 large onions, finely chopped
- 4 large tomatoes, pureed
- 1/2 cup cream
- 8 cashews
- 1.5 tbsp ginger garlic paste
- 2 tsp kashmiri chili powder
- 1.5 tsp tikka masala
- 1 tsp coriander powder
- 1/2 tsp turmeric
- 1 tsp garam masala
- 1 tbsp kasuri methi
- Salt to taste
MARINADE:
- 3/4 cup thick yogurt
- 2 tsp tikka masala
- 1 tsp red chili powder
- 1 tbsp lemon juice
- 1 tbsp oil
- Salt to taste
METHOD:
1. Cube chicken, mix with marinade ingredients. Marinate minimum 4 hours.
2. Thread onto skewers and grill on high heat until charred spots appear.
3. Alternatively, broil in oven at 220C for 15 minutes, turning once.
4. For sauce: heat butter and oil. Cook onions until deep golden, 20 minutes.
5. Add ginger garlic paste, cook 3 minutes. Add tomatoes and cashews.
6. Cook until oil separates thoroughly, about 15 minutes.
7. Cool mixture and blend until completely smooth. Strain through sieve.
8. Return to pan, add all spice powders. Cook 5 minutes.
9. Add cream, stir. Add grilled chicken tikka pieces.
10. Simmer 10 minutes. Add kasuri methi and garam masala.
TIPS: Straining the gravy through a fine mesh sieve gives the silky restaurant texture.
The char on tikka pieces is essential flavor, never skip the grilling step.
[/RECIPE]

[RECIPE]
NAME: Kheer (Rice Pudding)
CATEGORY: Dessert, North Indian
SERVES: 6
TIME: 1 hour
INGREDIENTS:
- 1.5 liters full fat milk
- 1/3 cup basmati rice
- 2/3 cup sugar (adjust to taste)
- 15 cashews, lightly toasted
- 15 almonds, blanched and sliced
- 2 tbsp golden raisins
- 1 tsp cardamom powder
- Generous pinch saffron
- 1 tbsp rose water
- 2 tbsp condensed milk (optional, for richness)
METHOD:
1. Wash and soak basmati rice for 30 minutes. Drain.
2. In a heavy bottomed pan, bring milk to a full boil stirring continuously.
3. Add soaked rice. Reduce to medium heat.
4. Cook stirring every 2 to 3 minutes to prevent sticking at bottom.
5. After 20 minutes add saffron soaked in 2 tbsp warm milk.
6. Continue cooking for another 20 minutes until milk thickens.
7. Rice grains should be completely soft and milk reduced to thick consistency.
8. Add sugar, cardamom powder and condensed milk if using. Cook 10 minutes.
9. Remove from heat. Add rose water, cashews, almonds and raisins.
10. Kheer thickens further as it cools. Adjust consistency with warm milk if needed.
TIPS: Never stop stirring during cooking or milk will burn at bottom.
Kheer served chilled is better than warm as flavors develop on resting.
Garnish with silver leaf (chandi ka vark) for special occasions.
[/RECIPE]

[RECIPE]
NAME: Aloo Paratha
CATEGORY: Breakfast, Punjabi, Vegetarian
SERVES: 4
TIME: 45 minutes
INGREDIENTS FOR DOUGH:
- 2 cups whole wheat flour (atta)
- 1/2 tsp salt
- Water to knead soft dough
- 1 tsp oil
INGREDIENTS FOR FILLING:
- 4 large potatoes, boiled and mashed
- 2 green chilies, finely chopped
- 1 inch ginger, grated
- 1 tbsp fresh coriander
- 1 tsp cumin powder
- 1/2 tsp red chili powder
- 1/2 tsp amchur
- 1/2 tsp ajwain (carom seeds)
- Salt to taste
- Butter and ghee for cooking
METHOD:
1. Knead atta with water and salt into smooth soft dough. Rest covered 30 minutes.
2. Mix all filling ingredients. Taste and adjust spicing. Divide into equal balls.
3. Roll small disc of dough. Place filling ball in center. Seal edges around filling.
4. Gently roll filled ball into flat paratha, about 7 inches diameter.
5. Cook on hot tawa (griddle) until small bubbles appear, flip.
6. Apply ghee generously on cooked side. Flip and apply ghee on other side.
7. Press gently with spatula. Cook until golden spots appear on both sides.
TIPS: The filling must be completely dry with no moisture or paratha will tear while rolling.
Cook on high heat for crispy exterior while keeping inside soft and fluffy.
Serve hot with white butter, yogurt and achaar (pickle).
[/RECIPE]

[RECIPE]
NAME: Fish Curry (Goan Style)
CATEGORY: Main Course, Goan, Coastal
SERVES: 4
TIME: 35 minutes
INGREDIENTS:
- 600g firm white fish (pomfret or kingfish), cut into steaks
- 1.5 cups fresh grated coconut
- 3 tbsp coconut oil
- 2 medium onions, sliced
- 2 tomatoes, chopped
- 4 cloves garlic
- 1 inch ginger
- 1 tsp turmeric
- 2 tsp Kashmiri chili powder
- 1 tsp coriander powder
- Marble sized ball of tamarind
- 2 green chilies, slit
- 10 curry leaves
- Salt to taste
METHOD:
1. Grind coconut with garlic, ginger, turmeric, chili and coriander to smooth paste.
2. Add tamarind and a little water, grind again. Extract thick coconut masala paste.
3. Heat coconut oil in wide pan. Add curry leaves and green chilies.
4. Add onions and cook until light golden.
5. Add tomatoes and cook until soft.
6. Add coconut masala paste. Cook on medium heat 5 minutes.
7. Add 1.5 cups water. Bring to boil.
8. Gently slide in fish steaks. Do not stir.
9. Cover and cook on medium heat 8 to 10 minutes. Fish should just flake.
10. Taste and adjust salt and sourness.
TIPS: Never overcook fish or it becomes rubbery. Use only coconut oil for authentic Goan flavor.
The curry tastes better after sitting for 30 minutes as flavors develop.
Serve with steamed red rice or boiled rice.
[/RECIPE]

[RECIPE]
NAME: Gajar Ka Halwa (Carrot Halwa)
CATEGORY: Dessert, North Indian
SERVES: 6
TIME: 1 hour
INGREDIENTS:
- 1 kg fresh red carrots, grated
- 1 liter full fat milk
- 3/4 cup sugar
- 4 tbsp ghee
- 1/2 cup khoya (mawa), crumbled
- 1 tsp cardamom powder
- 15 cashews
- 15 almonds, sliced
- 2 tbsp raisins
METHOD:
1. Heat ghee in a heavy wide pan. Add grated carrots.
2. Cook on high heat for 10 minutes, stirring continuously until moisture evaporates.
3. Add milk. Cook on medium heat stirring frequently.
4. Continue until milk is almost completely absorbed, about 30 to 40 minutes.
5. Add sugar. Mix well. Sugar will release moisture, continue cooking.
6. Add khoya, cardamom, nuts and raisins.
7. Cook on medium heat until halwa leaves sides of pan and ghee appears.
8. Total cooking time approximately 50 to 60 minutes.
TIPS: Use fresh red winter carrots (Delhi carrots) for best color and sweetness.
Halwa should not be dry but have a slight moisture and sheen from ghee.
Serve warm garnished with extra silver leaves and chopped pistachios.
[/RECIPE]
'''

# Repeat to build large enough training corpus
text = (BASE_CORPUS * 8).strip()
print(f'Total characters: {len(text):,}')
print(f'Estimated tokens: ~{len(text)//4:,}')
print(f'\nVocab sample (first 300 chars):\n{text[:300]}')


## 3. Tokenizer

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'Vocabulary size: {vocab_size}')

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

import torch
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f'Train tokens: {len(train_data):,}  |  Val tokens: {len(val_data):,}')


## 4. Hyperparameters — Upgraded for quality

In [ ]:
# ============================================================
# UPGRADED HYPERPARAMETERS
# v1 → v2 changes:
#   n_embd:  384  → 512   (+33%)
#   n_head:  6    → 8
#   n_layer: 6    → 8     (deeper model)
#   block_size: 256 → 384 (longer context)
#   max_iters: 5000 → 10000
#   batch_size: 64 → 128  (with grad accumulation)
# ============================================================

batch_size    = 64       # per step (accumulate 2 steps = effective 128)
accum_steps   = 2        # gradient accumulation
block_size    = 384      # longer context window
max_iters     = 10000    # 2x more training
eval_interval = 500
eval_iters    = 200
learning_rate = 3e-4
min_lr        = 3e-5     # cosine decay floor
warmup_iters  = 500      # linear warmup
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Bigger model
n_embd   = 512
n_head   = 8
n_layer  = 8
dropout  = 0.1   # lower dropout for bigger model

print(f'Device         : {device}')
print(f'Block size     : {block_size}')
print(f'Model size     : {n_embd}d / {n_head}h / {n_layer}L')
print(f'Effective batch: {batch_size * accum_steps}')
print(f'Training steps : {max_iters:,}')


In [ ]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def get_lr(step):
    """Cosine annealing with linear warmup."""
    if step < warmup_iters:
        return learning_rate * step / warmup_iters
    if step > max_iters:
        return min_lr
    progress = (step - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_lr + coeff * (learning_rate - min_lr)


## 5. Model — Upgraded Architecture

In [ ]:
import torch.nn as nn
from torch.nn import functional as F
import math

torch.manual_seed(42)

class CausalSelfAttention(nn.Module):
    """Efficient multi-head self-attention (single fused kernel)."""
    def __init__(self):
        super().__init__()
        assert n_embd % n_head == 0
        # Fused QKV projection
        self.c_attn  = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj  = nn.Linear(n_embd, n_embd,     bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer('bias', torch.tril(torch.ones(block_size, block_size))
                                          .view(1, 1, block_size, block_size))
        self.n_head = n_head
        self.n_embd = n_embd

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        hs = C // self.n_head
        # Reshape to (B, heads, T, head_size)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        # Scaled dot product attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(hs))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = self.attn_drop(F.softmax(att, dim=-1))
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_fc   = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=False)
        self.drop   = nn.Dropout(dropout)
        self.act    = nn.GELU()

    def forward(self, x):
        return self.drop(self.c_proj(self.act(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp  = MLP()

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class RecipeGPTv2(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(vocab_size, n_embd),
            wpe = nn.Embedding(block_size, n_embd),
            drop = nn.Dropout(dropout),
            h   = nn.ModuleList([Block() for _ in range(n_layer)]),
            ln_f = nn.LayerNorm(n_embd),
        ))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying
        self.apply(self._init_weights)
        # Scale residual projections (GPT-2 paper)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.transformer.drop(
            self.transformer.wte(idx) + self.transformer.wpe(pos)
        )
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :] / temperature
            # Top-K filtering
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            # Top-P (nucleus) filtering
            if top_p is not None:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_to_remove = cum_probs - F.softmax(sorted_logits, dim=-1) > top_p
                sorted_logits[sorted_to_remove] = float('-inf')
                logits = torch.zeros_like(logits).scatter_(1, sorted_idx, sorted_logits)
            idx_next = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = RecipeGPTv2().to(device)
params = sum(p.numel() for p in model.parameters())
print(f'RecipeGPT v2 ready!')
print(f'Parameters: {params/1e6:.1f}M')
print(f'Device    : {device}')


## 6. Training with Gradient Accumulation & LR Warmup

In [ ]:
import time, math

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.95),   # GPT-3 style betas
    weight_decay=0.1,    # weight decay for regularization
    eps=1e-8
)

train_losses, val_losses, loss_steps = [], [], []
start = time.time()

optimizer.zero_grad(set_to_none=True)

for step in range(max_iters):
    # Update learning rate
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Evaluate periodically
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss(model)
        elapsed = time.time() - start
        print(f'step {step:>5d} | train {losses["train"]:.4f} | val {losses["val"]:.4f} | lr {lr:.2e} | {elapsed:.0f}s')
        train_losses.append(losses['train'].item())
        val_losses.append(losses['val'].item())
        loss_steps.append(step)

    # Gradient accumulation
    for micro_step in range(accum_steps):
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)
        loss = loss / accum_steps
        loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

print(f'\nTraining complete in {(time.time()-start)/60:.1f} minutes')


## 7. Loss Curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(loss_steps, train_losses, 'b-o', label='Train Loss', markersize=4)
ax.plot(loss_steps, val_losses,   'r-o', label='Val Loss',   markersize=4)
ax.set_xlabel('Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('RecipeGPT v2 — Training Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final train loss: {train_losses[-1]:.4f}')
print(f'Final val loss  : {val_losses[-1]:.4f}')
print(f'Gap (overfit indicator): {val_losses[-1] - train_losses[-1]:.4f}')


## 8. Generate & Evaluate Quality

In [ ]:
def generate_recipe(prompt='[RECIPE]\nNAME:', max_tokens=600,
                     temperature=0.8, top_k=50, top_p=0.92):
    model.eval()
    safe = ''.join(c for c in prompt if c in stoi)
    ctx = torch.tensor([encode(safe)], dtype=torch.long, device=device)
    out = model.generate(ctx, max_new_tokens=max_tokens,
                         temperature=temperature, top_k=top_k, top_p=top_p)
    return decode(out[0].tolist())

# Test 1: Structured prompt
print('=' * 70)
print('TEST 1: Structured prompt — [RECIPE] NAME:')
print('=' * 70)
print(generate_recipe('[RECIPE]\nNAME: Paneer', temperature=0.75, top_k=40))


In [ ]:
# Test 2: Different dish types
prompts = [
    ('[RECIPE]\nNAME: Chicken', 'Chicken dish'),
    ('[RECIPE]\nNAME: Dal',     'Dal recipe'),
    ('[RECIPE]\nNAME: Lamb',    'Lamb curry'),
]
for prompt, label in prompts:
    print('=' * 70)
    print(f'TEST: {label}')
    print('=' * 70)
    print(generate_recipe(prompt, max_tokens=500, temperature=0.8, top_k=50, top_p=0.9))
    print()


In [ ]:
# Test 3: Dessert generation
print('=' * 70)
print('TEST: Dessert generation')
print('=' * 70)
print(generate_recipe('[RECIPE]\nNAME: Halwa', max_tokens=500, temperature=0.7, top_k=40))


In [ ]:
# ============================================================
# INTERACTIVE CELL — Change these!
# ============================================================

MY_DISH       = 'Biryani'    # What recipe to generate
MY_TEMP       = 0.8          # 0.6=focused, 1.0=balanced, 1.3=creative
MY_TOP_K      = 50
MY_TOP_P      = 0.92         # Nucleus sampling (0.9-0.95 works well)
MY_MAX_TOKENS = 700

result = generate_recipe(
    f'[RECIPE]\nNAME: {MY_DISH}',
    max_tokens=MY_MAX_TOKENS,
    temperature=MY_TEMP,
    top_k=MY_TOP_K,
    top_p=MY_TOP_P
)
print(result)


## 9. Save Model

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': vocab_size,
    'stoi': stoi,
    'itos': itos,
    'hyperparams': dict(
        n_embd=n_embd, n_head=n_head, n_layer=n_layer,
        block_size=block_size, dropout=dropout
    ),
    'train_losses': train_losses,
    'val_losses'  : val_losses,
    'version'     : 'v2'
}, 'recipe_gpt_v2.pt')
print('Saved: recipe_gpt_v2.pt')

import os
size_mb = os.path.getsize('recipe_gpt_v2.pt') / 1e6
print(f'File size: {size_mb:.1f} MB')
